# Part 9 Synthetic Cable-Length and Loss Characterization

This notebook builds a **fully synthetic** model for two experiments on the horn-to-SDR coaxial cable:

1. **Experiment 1 (square-wave reflectometry):** estimate round-trip delay from reflected wavelets and show why high square-wave frequency can cause incorrect wavelet pairing.
2. **Experiment 2 (fixed RF tone, two LO settings):** characterize power loss vs distance at the HI line, calibrate LO-dependent SDR response, infer unknown cable length, and perform a model-based indirect propagation-speed estimate.

The goal is a mathematically complete workflow that can later be mapped onto real measurements.

## Shared Transmission-Line Model

We use the following symbols:

$$
Z_0 = 50\,\Omega,\qquad \Gamma_L = \frac{Z_L - Z_0}{Z_L + Z_0},\qquad v_p = \mathrm{VF}\,c,
$$

$$
\tau = \frac{L}{v_p},\qquad \tau_{\mathrm{rt}} = \frac{2L}{v_p},\qquad
\gamma(\omega) = \alpha(\omega) + i\beta(\omega),\qquad \beta(\omega)=\frac{\omega}{v_p}.
$$

Power loss in dB for a one-way cable section of length $L$ at fixed RF frequency is written as

$$
P_{\mathrm{out,dBm}}(L)=P_{\mathrm{in,dBm}}-\alpha_{\mathrm{dB/m}}L.
$$

In this notebook we explicitly distinguish:
- **One-way delay** $\tau$ versus **round-trip delay** $\tau_{\mathrm{rt}}$.
- **Voltage reflection wavelets** (Experiment 1) versus **power attenuation** (Experiment 2).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal, optimize

plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["axes.grid"] = True

rng = np.random.default_rng(121)

C0 = 299_792_458.0
MU0 = 4.0 * np.pi * 1e-7
EPS0 = 8.854_187_8128e-12

Z0 = 50.0
F_RF_HI = 1_420.405_751_77e6
LO_1420 = 1_420.0e6
LO_1421 = 1_421.0e6

L_TRUE_M = 50.0
VF_TRUE = 0.66
VP_TRUE = VF_TRUE * C0
TAU_RT_TRUE = 2.0 * L_TRUE_M / VP_TRUE
AMP_RT_TRUE = 0.78

print(f"Round-trip delay true value: {TAU_RT_TRUE*1e9:.2f} ns")
print(f"One-way propagation speed true value: {VP_TRUE/1e8:.3f}e8 m/s")

In [ ]:
def reflection_coefficient(z_load, z0=50.0):
    return (z_load - z0) / (z_load + z0)


def propagation_speed(vf, c=C0):
    return vf * c


def attenuation_db_per_m(f_hz, alpha0_db_per_m, f0_hz, beta_exp):
    return alpha0_db_per_m * (f_hz / f0_hz) ** beta_exp


def square_wave_trace(t, f_sq_hz, amplitude_v, duty=0.5):
    return amplitude_v * signal.square(2.0 * np.pi * f_sq_hz * t, duty=duty)


def reflected_wavelet_trace(t, f_sq_hz, tau_rt_s, gamma_l, amp_rt, amplitude_v=1.0):
    incident = square_wave_trace(t, f_sq_hz, amplitude_v)
    reflected = gamma_l * amp_rt * square_wave_trace(t - tau_rt_s, f_sq_hz, amplitude_v)
    return incident + reflected, incident, reflected


def pair_wavelet_delay(t_incident, t_reflected, period_s, m=0):
    return t_reflected - (t_incident + m * period_s / 2.0)


def square_wave_harmonic_response(f0_hz, tau_rt_s, gamma_l, amp_rt, n_harmonics=20):
    n = np.arange(1, 2 * n_harmonics, 2)
    f_h = n * f0_hz
    response = 1.0 + gamma_l * amp_rt * np.exp(-1j * 2.0 * np.pi * f_h * tau_rt_s)
    return n, f_h, response


def baseband_offset_hz(f_rf_hz, lo_hz):
    return f_rf_hz - lo_hz


def counts_model(length_m, alpha_db_per_m, gain_linear, p0_dbm, noise_frac=0.01, rng_obj=None):
    if rng_obj is None:
        rng_obj = np.random.default_rng(0)
    length_arr = np.asarray(length_m, dtype=float)
    p_dbm = p0_dbm - alpha_db_per_m * length_arr
    counts_clean = gain_linear * 10.0 ** (p_dbm / 10.0)
    noise = rng_obj.normal(0.0, noise_frac, size=counts_clean.shape) * counts_clean
    counts = np.clip(counts_clean + noise, 1e-12, None)
    return counts, p_dbm


def joint_two_lo_attenuation_fit(lengths_m, counts_lo1, counts_lo2):
    lengths_m = np.asarray(lengths_m, dtype=float)
    y1 = 10.0 * np.log10(np.asarray(counts_lo1, dtype=float))
    y2 = 10.0 * np.log10(np.asarray(counts_lo2, dtype=float))

    n = lengths_m.size
    y = np.concatenate([y1, y2])
    x = np.zeros((2 * n, 3), dtype=float)
    x[:n, 0] = 1.0
    x[:n, 2] = -lengths_m
    x[n:, 1] = 1.0
    x[n:, 2] = -lengths_m

    params, _, _, _ = np.linalg.lstsq(x, y, rcond=None)
    y_hat = x @ params
    dof = y.size - params.size
    sigma2 = np.sum((y - y_hat) ** 2) / dof
    cov = sigma2 * np.linalg.inv(x.T @ x)

    return {
        "intercepts_db": params[:2],
        "alpha_db_per_m": params[2],
        "cov": cov,
        "y_hat": y_hat,
    }


def unknown_length_from_power(power_db, intercept_db, alpha_db_per_m):
    return (intercept_db - power_db) / alpha_db_per_m


def rg58_like_alpha_db_per_m(f_hz, epsilon_eff, tan_delta, sigma_s_per_m, a_m=0.46e-3, b_m=1.49e-3):
    mu = MU0
    eps = EPS0 * epsilon_eff

    rs = np.sqrt(np.pi * f_hz * mu / sigma_s_per_m)
    z0_eff = (1.0 / (2.0 * np.pi)) * np.sqrt(mu / eps) * np.log(b_m / a_m)
    r_per_m = rs / (2.0 * np.pi) * ((1.0 / a_m) + (1.0 / b_m))

    alpha_c_np_m = r_per_m / (2.0 * z0_eff)
    beta = 2.0 * np.pi * f_hz * np.sqrt(mu * eps)
    alpha_d_np_m = 0.5 * beta * tan_delta

    alpha_db_m = (alpha_c_np_m + alpha_d_np_m) * 8.685889638
    return alpha_db_m


def alpha_to_vp_rg58_like(alpha_db_per_m, f_hz, tan_delta, sigma_s_per_m, a_m=0.46e-3, b_m=1.49e-3, bracket=(1.2, 4.5)):
    def residual(epsilon_eff):
        return rg58_like_alpha_db_per_m(
            f_hz=f_hz,
            epsilon_eff=epsilon_eff,
            tan_delta=tan_delta,
            sigma_s_per_m=sigma_s_per_m,
            a_m=a_m,
            b_m=b_m,
        ) - alpha_db_per_m

    lo, hi = bracket
    r_lo = residual(lo)
    r_hi = residual(hi)

    if r_lo * r_hi <= 0.0:
        sol = optimize.root_scalar(residual, bracket=[lo, hi], method="brentq")
        eps_est = sol.root
    else:
        eps_est = lo if abs(r_lo) < abs(r_hi) else hi

    vp_est = C0 / np.sqrt(eps_est)
    return vp_est, eps_est

## Experiment 1 \u2014 Reflectometry with Square Waves

### Time-domain wavelet model

For a first-order reflection observed at the injection end,

$$
v_{\mathrm{obs}}(t)=v_{\mathrm{inc}}(t)+\Gamma_L A_{\mathrm{rt}}\,v_{\mathrm{inc}}(t-\tau_{\mathrm{rt}}),
$$

so the direct estimator is

$$
L = \frac{v_p\,\Delta t}{2},\qquad v_p = \frac{2L}{\Delta t},\qquad \Delta t=\tau_{\mathrm{rt}}.
$$

### Harmonic phase view

Writing the square wave as odd harmonics with fundamental $f_0$ gives, at harmonic frequency $f_n=n f_0$,

$$
V_{n,\mathrm{obs}}=V_n\left[1+\Gamma_L A_{\mathrm{rt}}\exp\left(-i2\pi f_n\tau_{\mathrm{rt}}\right)\right].
$$

This is equivalent physics, but a single phase measurement is modulo $2\pi$; practical recovery needs correct wavelet/harmonic pairing.

In [ ]:
# Termination behavior at a safely low square-wave frequency.
f_crit = 1.0 / (2.0 * TAU_RT_TRUE)
f_low = 0.10 * f_crit

t = np.linspace(0.0, 8.0 * TAU_RT_TRUE, 6000)
v_open, v_inc, v_ref_open = reflected_wavelet_trace(
    t=t,
    f_sq_hz=f_low,
    tau_rt_s=TAU_RT_TRUE,
    gamma_l=+1.0,
    amp_rt=AMP_RT_TRUE,
)
v_short, _, v_ref_short = reflected_wavelet_trace(
    t=t,
    f_sq_hz=f_low,
    tau_rt_s=TAU_RT_TRUE,
    gamma_l=-1.0,
    amp_rt=AMP_RT_TRUE,
)
v_match, _, _ = reflected_wavelet_trace(
    t=t,
    f_sq_hz=f_low,
    tau_rt_s=TAU_RT_TRUE,
    gamma_l=0.0,
    amp_rt=AMP_RT_TRUE,
)

np.testing.assert_allclose(v_match, v_inc, atol=1e-12)
np.testing.assert_allclose(v_ref_open, -v_ref_short, atol=1e-12)

plt.figure(figsize=(10, 5))
plt.plot(t * 1e9, v_inc, "k--", lw=1.4, label="Incident")
plt.plot(t * 1e9, v_open, lw=1.8, label="Observed (open, $\\Gamma_L=+1$)")
plt.plot(t * 1e9, v_short, lw=1.8, label="Observed (short, $\\Gamma_L=-1$)")
plt.plot(t * 1e9, v_match, lw=1.8, label="Observed (matched, $\\Gamma_L=0$)")
plt.axvline(TAU_RT_TRUE * 1e9, color="tab:red", ls=":", lw=1.5, label="$\\tau_{rt}$")
plt.xlabel("Time [ns]")
plt.ylabel("Voltage [arb]")
plt.title("Experiment 1A: Reflected wavelets for open/short/matched terminations")
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
# Harmonic phase behavior for the same cable delay.
n_h, f_h, h_resp = square_wave_harmonic_response(
    f0_hz=f_low,
    tau_rt_s=TAU_RT_TRUE,
    gamma_l=+1.0,
    amp_rt=AMP_RT_TRUE,
    n_harmonics=25,
)
phase_wrapped = np.angle(h_resp)
phase_unwrapped = np.unwrap(phase_wrapped)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), constrained_layout=True)
axes[0].plot(n_h, np.abs(h_resp), marker="o")
axes[0].set_xlabel("Odd harmonic index n")
axes[0].set_ylabel(r"$|1+\\Gamma A_{rt}e^{-i2\\pi f_n\\tau_{rt}}|$")
axes[0].set_title("Experiment 1B: Harmonic magnitude")

axes[1].plot(n_h, phase_wrapped, "o", label="Wrapped phase")
axes[1].plot(n_h, phase_unwrapped, "-", lw=1.4, label="Unwrapped phase")
axes[1].set_xlabel("Odd harmonic index n")
axes[1].set_ylabel("Phase [rad]")
axes[1].set_title("Wrapped vs unwrapped phase (2$\\pi$ ambiguity)")
axes[1].legend(loc="best")
plt.show()

### Experiment 1C \u2014 High-frequency wavelet-pairing failure mode

Adjacent square-wave transitions are separated by $T/2$. If $T/2$ becomes comparable to or smaller than $\tau_{\mathrm{rt}}$, then a reflected edge wavelet can be paired with the **wrong incident edge wavelet**.

Using incident-edge index shift $m$,

$$
\Delta t_{\mathrm{meas}}=\tau_{\mathrm{rt}}-m\frac{T}{2},\qquad
L_{\mathrm{meas}}=\frac{v_p\Delta t_{\mathrm{meas}}}{2}.
$$

The physically correct pairing is $m=0$. Wrong pairing ($m\ge1$) aliases the inferred length.

In [ ]:
freq_cases = {
    "Low (safe)": 0.10 * f_crit,
    "Near limit": 0.70 * f_crit,
    "High (aliased)": 1.50 * f_crit,
}

records = []
fig, axes = plt.subplots(3, 1, figsize=(11, 10), sharex=False)

for ax, (label, f_sq) in zip(axes, freq_cases.items()):
    period = 1.0 / f_sq
    half_period = period / 2.0
    t_end = max(3.0 * TAU_RT_TRUE + 2.0 * half_period, 3.0 * period)
    t_loc = np.linspace(0.0, t_end, 7000)

    obs, inc, _ = reflected_wavelet_trace(
        t=t_loc,
        f_sq_hz=f_sq,
        tau_rt_s=TAU_RT_TRUE,
        gamma_l=+1.0,
        amp_rt=AMP_RT_TRUE,
    )

    m_alias = int(np.floor(TAU_RT_TRUE / half_period))
    dt_correct = pair_wavelet_delay(0.0, TAU_RT_TRUE, period, m=0)
    dt_wrong = pair_wavelet_delay(0.0, TAU_RT_TRUE, period, m=m_alias)
    if dt_wrong <= 0.0:
        dt_wrong += half_period

    l_correct = VP_TRUE * dt_correct / 2.0
    l_wrong = VP_TRUE * dt_wrong / 2.0

    records.append(
        {
            "label": label,
            "f_sq_hz": f_sq,
            "half_period_ns": half_period * 1e9,
            "m_alias": m_alias,
            "l_correct_m": l_correct,
            "l_wrong_m": l_wrong,
            "error_wrong_pct": 100.0 * (l_wrong - L_TRUE_M) / L_TRUE_M,
        }
    )

    ax.plot(t_loc * 1e9, inc, "k--", lw=1.2, label="Incident")
    ax.plot(t_loc * 1e9, obs, lw=1.7, label="Observed")
    ax.axvline(0.0, color="tab:green", ls=":", lw=1.2, label="Chosen incident")
    ax.axvline(TAU_RT_TRUE * 1e9, color="tab:red", ls=":", lw=1.2, label="Reflected match")

    if m_alias >= 1:
        ax.axvline(m_alias * half_period * 1e9, color="tab:orange", ls="--", lw=1.3, label="Mispaired incident")

    ax.set_title(
        f"{label}: f = {f_sq/1e6:.3f} MHz, T/2 = {half_period*1e9:.1f} ns, "
        f"L(correct)={l_correct:.2f} m, L(wrong)={l_wrong:.2f} m"
    )
    ax.set_xlabel("Time [ns]")
    ax.set_ylabel("Voltage [arb]")
    ax.legend(loc="upper right")

plt.tight_layout()
plt.show()

print("Wavelet-pairing summary")
print("label | f_MHz | T/2_ns | m_alias | L_correct_m | L_wrong_m | wrong_error_%")
for rec in records:
    print(
        f"{rec['label']:>13} | {rec['f_sq_hz']/1e6:6.3f} | {rec['half_period_ns']:8.2f} |"
        f" {rec['m_alias']:7d} | {rec['l_correct_m']:11.3f} | {rec['l_wrong_m']:9.3f} | {rec['error_wrong_pct']:12.2f}"
    )

low_case = records[0]
high_case = records[-1]
assert abs(low_case["l_correct_m"] - L_TRUE_M) / L_TRUE_M < 0.01
assert abs(high_case["l_wrong_m"] - L_TRUE_M) / L_TRUE_M > 0.20

**Interpretation:** At low $f_{sq}$, edge wavelets are well separated and pairing is unambiguous. As $f_{sq}$ increases, $T/2$ shrinks and reflected wavelets overlap the timing grid of neighboring incident edges. If the phase or delay is read across the wrong wavelet pair, the inferred cable length aliases to an incorrect value.

## Experiment 2 \u2014 Fixed RF Tone, Two LO Settings, Variable Cable Length

We do **not** sweep frequency.

- RF tone is fixed at the HI rest frequency:
  $$f_{RF}=1420.40575177\ \mathrm{MHz}.$$
- SDR LO settings are:
  $$LO_1=1420\ \mathrm{MHz},\qquad LO_2=1421\ \mathrm{MHz}.$$
- Baseband offsets are therefore:
  $$\Delta f_1 = f_{RF}-LO_1,\qquad \Delta f_2 = f_{RF}-LO_2.$$

For reference cable lengths $L_i$, we simulate ADC-power proxy counts $C_{\ell}(L_i)$ with LO-dependent gain factors but a shared cable-loss slope:

$$
10\log_{10}C_{\ell}(L)=B_{\ell}-\alpha_{HI}L+\epsilon,
$$

where $\alpha_{HI}$ is the one-way power attenuation in dB/m at the HI frequency.

In [ ]:
# Synthetic experiment settings for fixed-tone attenuation calibration.
EPSILON_EFF_TRUE = 2.30
TAN_DELTA_ASSUMED = 2.0e-4
SIGMA_CU = 5.8e7
A_INNER_M = 0.46e-3
B_OUTER_M = 1.49e-3

ALPHA_HI_TRUE = rg58_like_alpha_db_per_m(
    f_hz=F_RF_HI,
    epsilon_eff=EPSILON_EFF_TRUE,
    tan_delta=TAN_DELTA_ASSUMED,
    sigma_s_per_m=SIGMA_CU,
    a_m=A_INNER_M,
    b_m=B_OUTER_M,
)

P0_DBM_TRUE = -20.0
L_REF_M = np.array([5, 10, 15, 20, 25, 30, 35, 40], dtype=float)
GAIN_LO_1420 = 2.0e8
GAIN_LO_1421 = 1.3e8

counts_1420, _ = counts_model(
    length_m=L_REF_M,
    alpha_db_per_m=ALPHA_HI_TRUE,
    gain_linear=GAIN_LO_1420,
    p0_dbm=P0_DBM_TRUE,
    noise_frac=0.010,
    rng_obj=rng,
)
counts_1421, _ = counts_model(
    length_m=L_REF_M,
    alpha_db_per_m=ALPHA_HI_TRUE,
    gain_linear=GAIN_LO_1421,
    p0_dbm=P0_DBM_TRUE,
    noise_frac=0.010,
    rng_obj=rng,
)

fit = joint_two_lo_attenuation_fit(L_REF_M, counts_1420, counts_1421)
B_hat_1420, B_hat_1421 = fit["intercepts_db"]
alpha_hat = fit["alpha_db_per_m"]

deltaB_true = 10.0 * np.log10(GAIN_LO_1420 / GAIN_LO_1421)
deltaB_hat = B_hat_1420 - B_hat_1421

df_1420_khz = baseband_offset_hz(F_RF_HI, LO_1420) / 1e3
df_1421_khz = baseband_offset_hz(F_RF_HI, LO_1421) / 1e3

print(f"Baseband offset for LO=1420 MHz: {df_1420_khz:.3f} kHz")
print(f"Baseband offset for LO=1421 MHz: {df_1421_khz:.3f} kHz")
print()
print(f"True alpha_HI:   {ALPHA_HI_TRUE:.4f} dB/m")
print(f"Fitted alpha_HI: {alpha_hat:.4f} dB/m")
print(f"True LO gain offset (dB):   {deltaB_true:.3f}")
print(f"Fitted LO gain offset (dB): {deltaB_hat:.3f}")

assert abs(alpha_hat - ALPHA_HI_TRUE) / ALPHA_HI_TRUE < 0.05
assert abs(deltaB_hat - deltaB_true) < 0.25

In [ ]:
# Plot raw counts-vs-length by LO and corrected shared attenuation line.
y1420 = 10.0 * np.log10(counts_1420)
y1421 = 10.0 * np.log10(counts_1421)
l_line = np.linspace(L_REF_M.min(), L_REF_M.max(), 300)

yfit_1420 = B_hat_1420 - alpha_hat * l_line
yfit_1421 = B_hat_1421 - alpha_hat * l_line

fig, axes = plt.subplots(1, 2, figsize=(12, 4.4), constrained_layout=True)
axes[0].scatter(L_REF_M, counts_1420, label="LO=1420 data")
axes[0].scatter(L_REF_M, counts_1421, label="LO=1421 data")
axes[0].plot(l_line, 10.0 ** (yfit_1420 / 10.0), lw=1.6, label="LO=1420 fit")
axes[0].plot(l_line, 10.0 ** (yfit_1421 / 10.0), lw=1.6, label="LO=1421 fit")
axes[0].set_xlabel("Reference cable length [m]")
axes[0].set_ylabel("ADC power proxy counts")
axes[0].set_yscale("log")
axes[0].set_title("Experiment 2 raw calibration data")
axes[0].legend(loc="best")

y_corr_1420 = y1420 - B_hat_1420
y_corr_1421 = y1421 - B_hat_1421
axes[1].scatter(L_REF_M, y_corr_1420, label="LO=1420 corrected")
axes[1].scatter(L_REF_M, y_corr_1421, label="LO=1421 corrected")
axes[1].plot(l_line, -alpha_hat * l_line, "k--", lw=1.5, label=f"Shared slope = -{alpha_hat:.3f} dB/m")
axes[1].set_xlabel("Reference cable length [m]")
axes[1].set_ylabel("Corrected power proxy [dB]")
axes[1].set_title("LO-normalized attenuation relation")
axes[1].legend(loc="best")

plt.show()

## Coax Attenuation and Speed-of-Light Link (Model-Based)

At fixed frequency, attenuation and phase constants follow from the low-loss RLGC model:

$$
\gamma = \sqrt{(R+i\omega L)(G+i\omega C)} = \alpha + i\beta,
$$

$$
\alpha \approx \frac{R}{2Z_0}+\frac{GZ_0}{2},\qquad
\beta \approx \omega\sqrt{LC}=\frac{\omega}{v_p}.
$$

For a coax line with effective dielectric constant $\epsilon_{\mathrm{eff}}$, one-way speed is

$$
v_p=\frac{c}{\sqrt{\epsilon_{\mathrm{eff}}}}.
$$

In this notebook we assume an **RG-58-like** geometry and copper conductivity, with a specified dielectric loss tangent. Under those assumptions we invert the fitted attenuation $\alpha_{HI}$ to estimate $\epsilon_{\mathrm{eff}}$ and hence $v_p$.

This is an **indirect, model-dependent** speed estimate. It is not as direct as Experiment 1 delay timing.

In [ ]:
# Unknown-length inference and model-based speed estimate from alpha_HI.
L_UNKNOWN_TRUE_M = 47.0

count_u_1420, _ = counts_model(
    length_m=np.array([L_UNKNOWN_TRUE_M]),
    alpha_db_per_m=ALPHA_HI_TRUE,
    gain_linear=GAIN_LO_1420,
    p0_dbm=P0_DBM_TRUE,
    noise_frac=0.006,
    rng_obj=rng,
)
count_u_1421, _ = counts_model(
    length_m=np.array([L_UNKNOWN_TRUE_M]),
    alpha_db_per_m=ALPHA_HI_TRUE,
    gain_linear=GAIN_LO_1421,
    p0_dbm=P0_DBM_TRUE,
    noise_frac=0.006,
    rng_obj=rng,
)

pow_u_1420_db = 10.0 * np.log10(count_u_1420[0])
pow_u_1421_db = 10.0 * np.log10(count_u_1421[0])

l_est_1420 = unknown_length_from_power(pow_u_1420_db, B_hat_1420, alpha_hat)
l_est_1421 = unknown_length_from_power(pow_u_1421_db, B_hat_1421, alpha_hat)
l_est_joint = 0.5 * (l_est_1420 + l_est_1421)

vp_est, eps_est = alpha_to_vp_rg58_like(
    alpha_db_per_m=alpha_hat,
    f_hz=F_RF_HI,
    tan_delta=TAN_DELTA_ASSUMED,
    sigma_s_per_m=SIGMA_CU,
    a_m=A_INNER_M,
    b_m=B_OUTER_M,
)
vp_true_model = C0 / np.sqrt(EPSILON_EFF_TRUE)

print(f"Unknown cable true length:         {L_UNKNOWN_TRUE_M:.3f} m")
print(f"Length estimate from LO=1420:      {l_est_1420:.3f} m")
print(f"Length estimate from LO=1421:      {l_est_1421:.3f} m")
print(f"Joint length estimate:             {l_est_joint:.3f} m")
print()
print(f"True epsilon_eff (model):          {EPSILON_EFF_TRUE:.4f}")
print(f"Estimated epsilon_eff from alpha:  {eps_est:.4f}")
print(f"True vp (model):                   {vp_true_model:.3e} m/s")
print(f"Estimated vp from alpha:           {vp_est:.3e} m/s")

assert abs(l_est_joint - L_UNKNOWN_TRUE_M) / L_UNKNOWN_TRUE_M < 0.05
assert abs(vp_est - vp_true_model) / vp_true_model < 0.08

eps_grid = np.linspace(1.6, 3.4, 300)
alpha_grid = [
    rg58_like_alpha_db_per_m(
        f_hz=F_RF_HI,
        epsilon_eff=e,
        tan_delta=TAN_DELTA_ASSUMED,
        sigma_s_per_m=SIGMA_CU,
        a_m=A_INNER_M,
        b_m=B_OUTER_M,
    )
    for e in eps_grid
]

plt.figure(figsize=(9.5, 4.8))
plt.plot(eps_grid, alpha_grid, lw=1.8, label=r"$\alpha_{model}(\epsilon_{eff})$")
plt.axhline(alpha_hat, color="tab:red", ls="--", lw=1.4, label=f"Fitted alpha = {alpha_hat:.3f} dB/m")
plt.axvline(eps_est, color="tab:green", ls=":", lw=1.4, label=f"Estimated epsilon_eff = {eps_est:.3f}")
plt.scatter([EPSILON_EFF_TRUE], [ALPHA_HI_TRUE], s=55, color="black", label="Synthetic truth")
plt.xlabel(r"$\epsilon_{eff}$")
plt.ylabel(r"$\alpha_{HI}$ [dB/m]")
plt.title("Experiment 2C: Indirect speed estimate from attenuation model")
plt.legend(loc="best")
plt.tight_layout()
plt.show()

## Final Notes and Equation Cheat Sheet

### Experiment 1 (direct delay)
$$
v_{obs}(t)=v_{inc}(t)+\Gamma_L A_{rt}v_{inc}(t-\tau_{rt}),\qquad
L=\frac{v_p\tau_{rt}}{2}.
$$

High $f_{sq}$ warning:
$$
\Delta t_{meas}=\tau_{rt}-m\frac{T}{2}\Rightarrow
L_{meas}=\frac{v_p\Delta t_{meas}}{2}.
$$
Wrong wavelet pairing ($m\ge1$) aliases cable length.

### Experiment 2 (fixed tone, two LO)
$$
10\log_{10}C_{\ell}(L)=B_{\ell}-\alpha_{HI}L+\epsilon
$$
with shared slope $\alpha_{HI}$ and LO-specific intercepts $B_{\ell}$.

Unknown length estimate:
$$
\hat L=\frac{B_{\ell}-10\log_{10}C_{\ell}}{\alpha_{HI}}.
$$

Model-based speed estimate (RG-58-like assumptions): infer $\epsilon_{eff}$ from fitted $\alpha_{HI}$, then
$$
v_p=\frac{c}{\sqrt{\epsilon_{eff}}}.
$$

Practical interpretation: Experiment 1 gives direct timing-based propagation information; Experiment 2 gives robust attenuation calibration and an indirect speed estimate under explicit cable-model assumptions.